In [ ]:
import pandas as pd
import numpy as np
import pickle
import yaml
import os
import shutil

label_lst = [i for i in range(10)]
noise_lst = [round(0.1*i,1) for i in range(1,11)]
algo_lst = ['denoising_dsvdd','smoothed_dsvdd','vanilla_dsvdd','ocsvm']
data_type_lst = ['cifar_1','cifar_2','mnist_1','mnist_2','cifar_yes','mnist_yes']

def open_yaml(yaml_path):
    with open(yaml_path) as f:
        film = yaml.load(f, Loader=yaml.FullLoader)
        return film
    
def check_condition(condition_gt,condition_each):
    
    false_dict = {}  
    for condition_k,condition_v in condition_gt.items():
        
        if condition_gt[condition_k] != condition_each.get(condition_k):
            false_dict[condition_k] = condition_each.get(condition_k)
            
    bool_value = True
    if len(false_dict.keys()) != 0:
        bool_value = False
            
    return false_dict, bool_value
    
def open_pkl(pkl_path):
    with open(pkl_path,'rb') as F:
        loaded_pkl= pickle.load(F)
    
    return loaded_pkl

def is_condition_valid(config_path,test_result_arr_path):
    
    if os.path.exists(config_path):
        loaded_yaml = open_yaml(config_path)
                
        condition_checked,bool_value = check_condition(
                condition_gt=each_condition,
                condition_each=loaded_yaml
            )
        
        if len(condition_checked.keys()) !=0:
            
            return False
    else:
        return False
        
    if os.path.exists(test_result_arr_path) is False:
        return False
    
    return True

def return_performance_per_filter_ratio(
    original_score,
    original_label,
    mask_arr_dict
):
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        confusion_matrix,
        precision_score,
        recall_score,
        f1_score,
    )
    performance_dict = {}
    for k,v in mask_arr_dict.items():
        
        masked_score= original_score[v]
        masked_label = original_label[v]
        
        # print(masked_score.shape,masked_label.shape)
        average_precision = average_precision_score(
            y_true=masked_label,
            y_score=masked_score
        )
        roc_auc = roc_auc_score(
            y_true=masked_label,
            y_score=masked_score
        )
        # tn,fp,fn,tp = confusion_matrix(
        #     y_true=masked_label,
        #     y_pred=masked_score
        # ).ravel()
        # precision = precision_score(
        #     y_true=masked_label,
        #     y_pred=masked_score
        # )
        # recall = recall_score(
        #     y_true=masked_label,
        #     y_pred=masked_score
        # )
        # f1 = f1_score(
        #     y_true=masked_label,
        #     y_pred=masked_score
        # )
        # performance_dict[k] = [average_precision,roc_auc,tn,fp,fn,tp,precision,recall,f1]
        performance_dict[k] = [average_precision,roc_auc]
        
    return performance_dict
        
def return_root_path_input_size_mask_key(algo,label,data_type):
    
    if data_type in ['cifar_1','cifar_2','mnist_1','mnist_2']:
                
        root_path = f'./history/{algo}_image_noised_new_start/'
    elif data_type in ['cifar_yes','mnist_yes']:
        root_path = f'./history/{algo}_image_non_noise_new_start/'
    else:
        raise Exception('wrong data_type 2')
    
    if data_type in ['cifar_1','cifar_2','cifar_yes']:
        input_size = 3072
        mask_arr_key = '___'.join(
            [
                'cifar',
                f'normal_label_{label}'
            ]
        )
            
    elif data_type in ['mnist_1','mnist_2','mnist_yes']:
        input_size = 784
        mask_arr_key = '___'.join(
            [
                'mnist',
                f'normal_label_{label}'
            ]
        )
    else:
        raise Exception('wrong data_type')
    
    return root_path,input_size,mask_arr_key
    
    
    
result_dict = {}
mask_idx_arr_dict = open_pkl('./label_idx_arr_per_ratio_image_ver.pkl')

for algo in algo_lst:
    for data_type in data_type_lst:
        for label in label_lst:
            
            root_path,input_size,mask_arr_key = return_root_path_input_size_mask_key(
                algo=algo,
                label=label,
                data_type=data_type
            )
            each_result_key = '___'.join(
                [
                    data_type,
                    f'normal_label_{label}'
                ]
            )
            result_dict[each_result_key] = {}
            
            if algo == 'denoising_dsvdd':
            
                for noise_value in noise_lst:
                    
                    result_dict[each_result_key][
                        f'{algo}___noise_ratio_{noise_value}'
                    ] = []
                                
                    each_condition = {
                        'data_type':data_type,
                        'normal_label':label,
                        'noise_ratio':noise_value,
                        'inputSize':input_size,
                        'do_zScore':True,
                        'which_model': algo,
                        'useMax_when_val':False,
                    }
                    upper_path = os.path.join(
                        root_path,
                        data_type,
                        f'normal_label_{label}',
                        f'noise_{noise_value}',
                        
                    )
                    if os.path.exists(upper_path):
                        flg = 0
                        under_path_lst = os.listdir(upper_path)
                        for under_path in under_path_lst:
                            
                            each_path = os.path.join(
                                upper_path,
                                under_path
                            )
                            config_path = os.path.join(
                                each_path,
                                'configs',
                                'resultConfig.yaml'
                            )
                            test_result_arr_path = os.path.join(
                                each_path,
                                'test_result_only.pkl'
                            )
                            if is_condition_valid(
                                config_path=config_path,
                                test_result_arr_path=test_result_arr_path
                            ):
                                
                                result_pkl = open_pkl(test_result_arr_path)
                                
                                performance_per_filter_ratio = return_performance_per_filter_ratio(
                                    original_score= result_pkl['test_score'],
                                    original_label=result_pkl['test_label'],
                                    mask_arr_dict = mask_idx_arr_dict[mask_arr_key]
                                )
                                if len(
                                    result_dict[each_result_key][
                                        f'{algo}___noise_ratio_{noise_value}'
                                    ]    
                                ) < 10:
                                    result_dict[each_result_key][
                                        f'{algo}___noise_ratio_{noise_value}'
                                    ].append(performance_per_filter_ratio)
                                
                            else:
                                continue
                            
            else:
                
                result_dict[each_result_key][
                        f'{algo}'
                    ] = []
                each_condition = {
                    'data_type':data_type,
                    'normal_label':label,
                    'inputSize':input_size,
                    'do_zScore':True,
                    'which_model': algo,
                    'useMax_when_val':False,
                }
                upper_path = os.path.join(
                    root_path,
                    data_type,
                    f'normal_label_{label}',
                )
                if os.path.exists(upper_path):
                    flg = 0
                    under_path_lst = os.listdir(upper_path)
                    for under_path in under_path_lst:
                        
                        
                        each_path = os.path.join(
                            upper_path,
                            under_path
                        )
                        config_path = os.path.join(
                            each_path,
                            'configs',
                            'resultConfig.yaml'
                        )
                        test_result_arr_path = os.path.join(
                            each_path,
                            'test_result_only.pkl'
                        )
                        if is_condition_valid(
                            config_path=config_path,
                            test_result_arr_path=test_result_arr_path
                        ):
                            
                            
                            result_pkl = open_pkl(test_result_arr_path)
                            
                            performance_per_filter_ratio = return_performance_per_filter_ratio(
                                original_score= result_pkl['test_score'],
                                original_label=result_pkl['test_label'],
                                mask_arr_dict = mask_idx_arr_dict[mask_arr_key]
                            )
                            if len(
                                    result_dict[each_result_key][
                                    f'{algo}'
                                ]
                            ) < 10:
                                
                                result_dict[each_result_key][
                                    f'{algo}'
                                ].append(performance_per_filter_ratio)
                            
                        else:
                            continue
                else:
                    print(f'there is no {upper_path}')    
            
            # print(data_type,label,noise_value,'  complete')    
            
            print(algo,data_type,label,' complete')

with open('./filter_performance_results_image_ver.pkl','wb') as F:
    pickle.dump(result_dict,F)
    
print('mission complete')

denoising_dsvdd cifar_1 0  complete
denoising_dsvdd cifar_1 1  complete
denoising_dsvdd cifar_1 2  complete
denoising_dsvdd cifar_1 3  complete
denoising_dsvdd cifar_1 4  complete
denoising_dsvdd cifar_1 5  complete
denoising_dsvdd cifar_1 6  complete
denoising_dsvdd cifar_1 7  complete
denoising_dsvdd cifar_1 8  complete
denoising_dsvdd cifar_1 9  complete
denoising_dsvdd cifar_2 0  complete
denoising_dsvdd cifar_2 1  complete
denoising_dsvdd cifar_2 2  complete
denoising_dsvdd cifar_2 3  complete
denoising_dsvdd cifar_2 4  complete
denoising_dsvdd cifar_2 5  complete
denoising_dsvdd cifar_2 6  complete
denoising_dsvdd cifar_2 7  complete
denoising_dsvdd cifar_2 8  complete
denoising_dsvdd cifar_2 9  complete
denoising_dsvdd mnist_1 0  complete
denoising_dsvdd mnist_1 1  complete
denoising_dsvdd mnist_1 2  complete
denoising_dsvdd mnist_1 3  complete
denoising_dsvdd mnist_1 4  complete
denoising_dsvdd mnist_1 5  complete
denoising_dsvdd mnist_1 6  complete
denoising_dsvdd mnist_1 7  c